In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

from src.preprocess import (
    select_core_features,
    clean_features,
    build_preprocessor
)

df = pd.read_csv("../data/home-credit-default-risk/application_train.csv")

df = select_core_features(df)

df = clean_features(df)

X = df.drop(columns=["TARGET"])
y = df["TARGET"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline = Pipeline([
    ("preprocess", build_preprocessor()),
    ("model", LogisticRegression(
        max_iter=5000,           # allow convergence
        class_weight="balanced",  # handle class imbalance
        solver="lbfgs"
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_val)
y_prob = pipeline.predict_proba(X_val)[:, 1]

print("Train Accuracy:", pipeline.score(X_train, y_train))
print("Validation Accuracy:", pipeline.score(X_val, y_val))

print("\nClassification Report:")
print(classification_report(y_val, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_val, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

Train Accuracy: 0.6859573672400897
Validation Accuracy: 0.6869746191242704

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.69      0.80     56538
           1       0.16      0.68      0.26      4965

    accuracy                           0.69     61503
   macro avg       0.56      0.68      0.53     61503
weighted avg       0.90      0.69      0.76     61503

ROC-AUC Score: 0.7446211955156612

Confusion Matrix:
[[38894 17644]
 [ 1608  3357]]
